# Binary Classification — Logistic Regression

---

**Project:** ML with Scikit-Learn  
**Notebook:** Binary Classification  
**Algorithm:** Logistic Regression  
**Dataset:** Play Golf Dataset  
**Author:** Ather-ops  

---

## Objective

Given weather conditions — Temperature, Humidity, and Wind — predict whether a person **will play golf or not**. This is a classic binary classification problem where the output is one of exactly two classes: `Yes` (play) or `No` (do not play).

---

## Linear Regression vs Logistic Regression

| Aspect | Linear Regression | Logistic Regression |
|--------|------------------|---------------------|
| Output | Continuous number (e.g. 450.5K price) | Probability between 0 and 1 |
| Prediction | Any real value | Class label (0 or 1) |
| Use case | Regression problems | Classification problems |
| Core function | $\hat{y} = wx + b$ | $\hat{p} = \sigma(wx + b)$ |
| Loss function | Mean Squared Error | Binary Cross-Entropy |

---

## The Sigmoid Function

Logistic Regression takes the linear output and passes it through the **sigmoid function** to squeeze any value into the range (0, 1):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The output $\sigma(z)$ is interpreted as the **probability of the positive class** (PlayGolf = Yes):
- If $\sigma(z) \geq 0.5$ → predict **Yes** (play golf)
- If $\sigma(z) < 0.5$ → predict **No** (do not play)

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Create dataset |
| 3 | Exploratory Data Analysis |
| 4 | Label Encoding — convert text to numbers |
| 5 | Descriptive statistics and missing value check |
| 6 | Feature and target selection |
| 7 | Train-test split |
| 8 | Feature scaling |
| 9 | Model training |
| 10 | Predictions and probabilities |
| 11 | Model evaluation — Accuracy, Precision, Recall, F1 |
| 12 | Confusion Matrix |
| 13 | Classification Report |
| 14 | ROC Curve and AUC Score |
| 15 | Feature importance — coefficients |
| 16 | Predict for a new weather condition |

---
## Step 1 — Import Libraries

We import everything needed for a full classification pipeline. Three new imports appear here that did not exist in the regression notebooks:

| Import | Role |
|--------|------|
| `LabelEncoder` | Converts text categories (`Hot`, `High`, `No`) into integers |
| `confusion_matrix` | 2x2 table showing TP, TN, FP, FN |
| `classification_report` | Precision, Recall, F1 per class |
| `roc_curve` | Computes the ROC curve points |
| `roc_auc_score` | Area Under the ROC Curve — overall classifier quality |
| `accuracy_score` | Fraction of correct predictions |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

---
## Step 2 — Create Dataset

We load the Play Golf dataset directly from a dictionary. It contains 10 weather observations and a binary decision for each.

| Column | Values | Type |
|--------|--------|------|
| `Temperature` | Hot, Mild, Cool | Categorical — 3 classes |
| `Humidity` | High, Normal | Categorical — 2 classes |
| `Windy` | Yes, No | Categorical — 2 classes |
| `PlayGolf` | Yes, No | Categorical — target variable, binary |

All four columns are text. Before any ML model can process this data, every column must be converted to numbers. That happens in Step 4.

In [ ]:
# Create the Play Golf dataset
data = {
    "Temperature": ["Hot",  "Hot",  "Mild", "Cool", "Cool",
                    "Mild", "Hot",  "Hot",  "Cool", "Mild"],
    "Humidity":    ["High", "High", "High", "Normal", "Normal",
                    "Normal", "Normal", "High", "High", "High"],
    "Windy":       ["No",   "Yes",  "No",   "No",    "Yes",
                    "No",   "No",   "Yes",  "No",    "Yes"],
    "PlayGolf":    ["No",   "No",   "Yes",  "Yes",   "No",
                    "Yes",  "Yes",  "No",   "No",    "Yes"]
}

df = pd.DataFrame(data)

print("="*60)
print("Play Golf Dataset — Raw (10 observations)")
print("="*60)
print(df.to_string(index=True))
print("="*60)
print("Shape:", df.shape)
print("="*60)

---
## Step 3 — Exploratory Data Analysis

Before encoding or modeling, we explore the raw distribution of each feature and the class balance of the target.

**Three questions we answer here:**

1. How is each feature distributed across its categories?
2. Is the target (`PlayGolf`) balanced? Imbalanced classes need special handling.
3. For each feature category, how often does golf get played?

A class imbalance exists when one class appears far more than the other (e.g. 90% No, 10% Yes). In that case, a model that always predicts No achieves 90% accuracy while learning nothing. With only 10 samples, we inspect counts carefully.

In [ ]:
# EDA — distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot 1: Target class distribution
play_counts = df["PlayGolf"].value_counts()
axes[0, 0].bar(play_counts.index, play_counts.values,
               color=["tomato", "steelblue"], edgecolor="white", width=0.5)
axes[0, 0].set_title("PlayGolf — Class Distribution", fontsize=12)
axes[0, 0].set_ylabel("Count")
for i, (label, val) in enumerate(play_counts.items()):
    axes[0, 0].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[0, 0].grid(True, axis="y", linestyle="--", alpha=0.5)

# Plot 2: Temperature distribution
temp_counts = df["Temperature"].value_counts()
axes[0, 1].bar(temp_counts.index, temp_counts.values,
               color="darkorange", edgecolor="white", width=0.5)
axes[0, 1].set_title("Temperature Distribution", fontsize=12)
axes[0, 1].set_ylabel("Count")
for i, val in enumerate(temp_counts.values):
    axes[0, 1].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[0, 1].grid(True, axis="y", linestyle="--", alpha=0.5)

# Plot 3: Humidity distribution
hum_counts = df["Humidity"].value_counts()
axes[1, 0].bar(hum_counts.index, hum_counts.values,
               color="seagreen", edgecolor="white", width=0.5)
axes[1, 0].set_title("Humidity Distribution", fontsize=12)
axes[1, 0].set_ylabel("Count")
for i, val in enumerate(hum_counts.values):
    axes[1, 0].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[1, 0].grid(True, axis="y", linestyle="--", alpha=0.5)

# Plot 4: Windy distribution
wind_counts = df["Windy"].value_counts()
axes[1, 1].bar(wind_counts.index, wind_counts.values,
               color="mediumpurple", edgecolor="white", width=0.5)
axes[1, 1].set_title("Windy Distribution", fontsize=12)
axes[1, 1].set_ylabel("Count")
for i, val in enumerate(wind_counts.values):
    axes[1, 1].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[1, 1].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Exploratory Data Analysis — Feature and Target Distributions",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("="*60)

---
## Step 3b — Play Golf Rate per Feature Category

We group each feature by its category and compute the fraction of `Yes` outcomes. This tells us which weather conditions are associated with playing golf most frequently.

For example, if all `Cool` temperature observations result in playing, then Temperature = Cool is a strong positive signal for the classifier.

In [ ]:
# Play rate per feature category
features_to_check = ["Temperature", "Humidity", "Windy"]

print("Play Golf Rate by Feature Category")
print("="*60)
for feat in features_to_check:
    rate = df.groupby(feat)["PlayGolf"].apply(
        lambda x: (x == "Yes").sum() / len(x) * 100
    ).round(1)
    print(f"\n{feat}:")
    for cat, pct in rate.items():
        bar = "|" * int(pct / 10)
        print(f"  {cat:10s}: {pct:5.1f}%  {bar}")
print("="*60)

---
## Step 4 — Label Encoding

Logistic Regression requires numerical inputs. We use `LabelEncoder` to convert every text column to integers.

**How LabelEncoder works:**  
It sorts the unique values alphabetically and assigns integer codes starting from 0.

| Column | Original values | Encoded values |
|--------|----------------|----------------|
| Temperature | Cool, Hot, Mild | 0, 1, 2 |
| Humidity | High, Normal | 0, 1 |
| Windy | No, Yes | 0, 1 |
| PlayGolf | No, Yes | 0, 1 |

We create a separate copy of the DataFrame (`df_encoded`) so the original text version remains available for reference. Each column gets its own `LabelEncoder` instance — we save them in a dictionary so we can reverse-encode predictions later if needed.

In [ ]:
# Label Encoding — convert all text columns to integers
df_encoded = df.copy()
encoders   = {}  # store each encoder for reference

for col in df_encoded.columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col]   = le
    print(f"Column '{col}'")
    print(f"  Original classes : {le.classes_.tolist()}")
    print(f"  Encoded as       : {list(range(len(le.classes_)))}")
    print()

print("="*60)
print("Encoded DataFrame:")
print("="*60)
print(df_encoded.to_string(index=True))
print("="*60)

---
## Step 5 — Descriptive Statistics and Missing Value Check

After encoding, we inspect the numerical DataFrame to confirm the encoding worked correctly and that no missing values exist.

**What to verify:**
- All columns now show integer values (0, 1, or 2)
- `count` is 10 for every column — no missing values introduced
- `PlayGolf` column (target) should have only values 0 and 1

In [ ]:
# Descriptive statistics after encoding
print("Descriptive Statistics — Encoded Dataset")
print("="*60)
print(df_encoded.describe().round(3))
print("="*60)
print("Missing Values:")
print(df_encoded.isnull().sum())
print("="*60)
print("Target class counts (PlayGolf):")
print(df_encoded["PlayGolf"].value_counts())
print("  0 = No (do not play)")
print("  1 = Yes (play)")
print("="*60)

---
## Step 6 — Feature and Target Selection

We separate the encoded DataFrame into the input feature matrix `X` and the target vector `Y`.

| Variable | Columns | Shape | Role |
|----------|---------|-------|------|
| `X` | Temperature, Humidity, Windy | (10, 3) | Input features — what the model sees |
| `Y` | PlayGolf | (10,) | Target — 0 = No, 1 = Yes |

Note that `PlayGolf` is dropped from `X`. We never include the target in the feature matrix — that would be data leakage.

In [ ]:
# Feature matrix and target vector
X = df_encoded.drop("PlayGolf", axis=1)
Y = df_encoded["PlayGolf"]

print("Feature and Target Selection")
print("="*60)
print("Feature matrix X shape :", X.shape)
print("Target vector Y shape  :", Y.shape)
print("="*60)
print("X — all rows:")
print(X.to_string())
print("="*60)
print("Y — all values:")
print(Y.to_string())
print("="*60)

---
## Step 7 — Train-Test Split

We split the 10 observations into training and test sets.

| Parameter | Value | Effect |
|-----------|-------|--------|
| `test_size=0.2` | 20% | 2 observations for testing |
| `random_state=42` | Fixed | Same split every run |
| `stratify=Y` | Target | Ensures both classes appear in both sets |

**Why `stratify=Y` matters here:**  
With only 10 samples, a random split could accidentally put all `Yes` examples in the training set and all `No` examples in the test set (or vice versa). `stratify=Y` ensures both classes are proportionally represented in both sets — essential for small datasets.

In [ ]:
# Train-test split with stratification
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

print("Train-Test Split")
print("="*60)
print(f"Total samples  : {len(X)}")
print(f"Training set   : {len(X_train)} samples")
print(f"Test set       : {len(X_test)} samples")
print("="*60)
print("Training target distribution:")
print(Y_train.value_counts().to_string())
print("\nTest target distribution:")
print(Y_test.value_counts().to_string())
print("  (0 = No, 1 = Yes)")
print("="*60)

---
## Step 8 — Feature Scaling

Even though the encoded values (0, 1, 2) are already small numbers, we apply StandardScaler for consistency and correctness.

**Why scale encoded categorical features?**  
Temperature has 3 levels (0, 1, 2) while Humidity has 2 levels (0, 1). Without scaling, Temperature contributes larger raw values and may be over-weighted relative to Humidity. Scaling puts all features on equal footing.

The same rule applies: **fit only on training data**, transform both.

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("StandardScaler Applied")
print("="*60)
print("Learned means per feature  :", scaler.mean_.round(4))
print("Learned scales per feature :", scaler.scale_.round(4))
print("Feature order              :", X.columns.tolist())
print("="*60)
print("X_train_scaled:")
print(pd.DataFrame(X_train_scaled,
                   columns=X.columns).round(4).to_string())
print("="*60)

---
## Step 9 — Model Training

We initialize `LogisticRegression` and train it on the scaled training data.

**What happens during `.fit()`:**

Logistic Regression finds weights $w_1, w_2, w_3$ and bias $b$ that minimize **Binary Cross-Entropy loss**:

$$L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

Where $\hat{p}_i = \sigma(w_1 x_1 + w_2 x_2 + w_3 x_3 + b)$ is the predicted probability of playing golf.

**Key parameters:**

| Parameter | Value | Meaning |
|-----------|-------|--------|
| `max_iter=1000` | 1000 | Allow enough iterations for convergence on this small dataset |
| `random_state=42` | 42 | Reproducible results |

In [ ]:
# Model training
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, Y_train)

feature_names = X.columns.tolist()

print("Logistic Regression Model Trained")
print("="*60)
print(f"Intercept (b) : {model.intercept_[0]:.4f}")
print("="*60)
print("Learned Weights:")
for feat, coef in zip(feature_names, model.coef_[0]):
    direction = "increases" if coef > 0 else "decreases"
    print(f"  {feat:15s}: {coef:+.4f}  ({direction} probability of playing)")
print("="*60)
print("Logistic equation:")
terms = " + ".join(
    f"{c:.4f}*{f}" for c, f in zip(model.coef_[0], feature_names)
)
print(f"  z = {terms} + {model.intercept_[0]:.4f}")
print(f"  P(PlayGolf=Yes) = sigmoid(z)")

---
## Step 10 — Predictions and Probabilities

Logistic Regression gives us two outputs:

| Method | Returns | Description |
|--------|---------|-------------|
| `model.predict()` | Class label (0 or 1) | Hard decision — the final answer |
| `model.predict_proba()` | [P(No), P(Yes)] | Soft probability for each class |

The probability is more informative than the hard label. A prediction of 0.51 (barely Yes) should be treated very differently from 0.99 (confidently Yes). We print both for every test observation.

In [ ]:
# Predictions and probabilities
y_pred       = model.predict(X_test_scaled)
y_prob       = model.predict_proba(X_test_scaled)
y_prob_yes   = y_prob[:, 1]   # probability of PlayGolf = Yes

# Decode back to text labels for readability
le_target    = encoders["PlayGolf"]
actual_text  = le_target.inverse_transform(Y_test.values)
pred_text    = le_target.inverse_transform(y_pred)

results = pd.DataFrame({
    "Temperature" : [encoders["Temperature"].classes_[v] for v in X_test["Temperature"].values],
    "Humidity"    : [encoders["Humidity"].classes_[v]    for v in X_test["Humidity"].values],
    "Windy"       : [encoders["Windy"].classes_[v]       for v in X_test["Windy"].values],
    "Actual"      : actual_text,
    "Predicted"   : pred_text,
    "P(No)"       : y_prob[:, 0].round(4),
    "P(Yes)"      : y_prob[:, 1].round(4),
    "Correct"     : ["Yes" if a == p else "No" for a, p in zip(actual_text, pred_text)]
})

print("Prediction Results — Test Set")
print("="*80)
print(results.to_string(index=False))
print("="*80)

---
## Step 11 — Model Evaluation

Classification is evaluated differently from regression. We use four metrics:

| Metric | Formula | Meaning |
|--------|---------|--------|
| Accuracy | $\frac{TP + TN}{Total}$ | Overall fraction of correct predictions |
| Precision | $\frac{TP}{TP + FP}$ | Of all predicted Yes, how many were actually Yes? |
| Recall | $\frac{TP}{TP + FN}$ | Of all actual Yes, how many did the model catch? |
| F1 Score | $\frac{2 \times Precision \times Recall}{Precision + Recall}$ | Harmonic mean of precision and recall |

**Abbreviations:**

| Term | Meaning |
|------|---------|
| TP | True Positive — predicted Yes, actual Yes |
| TN | True Negative — predicted No, actual No |
| FP | False Positive — predicted Yes, actual No (false alarm) |
| FN | False Negative — predicted No, actual Yes (missed case) |

For a golf prediction model, a **False Negative** (missed a play day) and **False Positive** (showed up when conditions were bad) have different costs. Precision and Recall capture these separately.

In [ ]:
# Model evaluation metrics
from sklearn.metrics import precision_score, recall_score, f1_score

accuracy  = accuracy_score(Y_test, y_pred)
precision = precision_score(Y_test, y_pred, zero_division=0)
recall    = recall_score(Y_test, y_pred, zero_division=0)
f1        = f1_score(Y_test, y_pred, zero_division=0)

print("Model Evaluation — Test Set")
print("="*60)
print(f"Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print("="*60)

---
## Step 12 — Confusion Matrix

The confusion matrix is a 2x2 table that shows exactly where the model made correct and incorrect predictions:

```
                  Predicted No    Predicted Yes
Actual No    |       TN       |       FP       |
Actual Yes   |       FN       |       TP       |
```

We visualize it as a colored grid with counts annotated in each cell. Cells on the diagonal (top-left and bottom-right) are correct predictions. Off-diagonal cells are errors.

In [ ]:
# Confusion matrix
cm = confusion_matrix(Y_test, y_pred)

print("Confusion Matrix:")
print("="*60)
print("                Predicted No    Predicted Yes")
print(f"Actual No    :      {cm[0,0]}                {cm[0,1]}")
print(f"Actual Yes   :      {cm[1,0]}                {cm[1,1]}")
print("="*60)
print(f"True Negatives  (TN): {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives  (TP): {cm[1,1]}")
print("="*60)

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)

labels = ["No (0)", "Yes (1)"]
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted No", "Predicted Yes"], fontsize=10)
ax.set_yticklabels(["Actual No", "Actual Yes"], fontsize=10)

# Annotate cells
cell_labels = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i,
                f"{cell_labels[i][j]}\n{cm[i, j]}",
                ha="center", va="center",
                fontsize=13, fontweight="bold",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

ax.set_title("Confusion Matrix", fontsize=13)
plt.tight_layout()
plt.show()
print("="*60)

---
## Step 13 — Classification Report

`classification_report` prints Precision, Recall, and F1 for **both classes** — not just the positive class. This is important because a model might perform well on one class but poorly on the other.

**Reading the report:**

| Row | Meaning |
|-----|---------|
| Class 0 (No) | Metrics for predicting No |
| Class 1 (Yes) | Metrics for predicting Yes |
| Macro avg | Simple average of both classes — treats them equally |
| Weighted avg | Average weighted by class count — accounts for imbalance |
| Support | Number of actual samples in each class |

In [ ]:
# Classification report
print("Classification Report")
print("="*60)
report = classification_report(
    Y_test, y_pred,
    target_names=["No (0)", "Yes (1)"]
)
print(report)
print("="*60)

---
## Step 14 — ROC Curve and AUC Score

The **ROC (Receiver Operating Characteristic) Curve** plots:
- X-axis: False Positive Rate (FPR) = FP / (FP + TN)
- Y-axis: True Positive Rate (TPR) = TP / (TP + FN)

It sweeps the classification threshold from 0 to 1 and plots how TPR and FPR change. A threshold of 0.5 is the standard cutoff but it can be adjusted based on the problem.

**AUC — Area Under the Curve:**

| AUC Value | Meaning |
|----------|---------|
| 1.0 | Perfect classifier |
| 0.9 – 1.0 | Excellent |
| 0.7 – 0.9 | Good |
| 0.5 – 0.7 | Weak |
| 0.5 | Random classifier — no better than a coin flip |
| Below 0.5 | Worse than random — predictions are inverted |

The diagonal dashed line represents a random classifier with AUC = 0.5. Any curve above this line indicates the model has learned something useful.

In [ ]:
# ROC Curve and AUC Score
fpr, tpr, thresholds = roc_curve(Y_test, y_prob_yes)
auc_score = roc_auc_score(Y_test, y_prob_yes)

print(f"AUC Score : {auc_score:.4f}")
print("="*60)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr,
         color="steelblue", linewidth=2.5,
         label=f"Logistic Regression (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1],
         color="gray", linestyle="--", linewidth=1.5,
         label="Random Classifier (AUC = 0.50)")
plt.fill_between(fpr, tpr, alpha=0.1, color="steelblue")

plt.xlabel("False Positive Rate (FPR)", fontsize=12)
plt.ylabel("True Positive Rate (TPR)", fontsize=12)
plt.title(f"ROC Curve — Play Golf Classifier  |  AUC = {auc_score:.4f}",
          fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*60)

---
## Step 15 — Feature Importance — Learned Coefficients

In Logistic Regression, the learned weights indicate each feature's contribution to the log-odds of the positive class (PlayGolf = Yes).

- **Positive coefficient** → higher value of that feature → higher probability of playing golf
- **Negative coefficient** → higher value → lower probability of playing golf
- **Larger absolute value** → stronger influence on the prediction

Since all features are scaled, the magnitude of each coefficient is directly comparable across features.

In [ ]:
# Feature importance — logistic regression coefficients
feature_names = X.columns.tolist()
coefs         = model.coef_[0]
colors        = ["steelblue" if c > 0 else "tomato" for c in coefs]

print("Learned Coefficients (scaled features):")
print("="*60)
for feat, coef in zip(feature_names, coefs):
    print(f"  {feat:15s}: {coef:+.4f}")
print("="*60)

plt.figure(figsize=(7, 4))
bars = plt.barh(feature_names, coefs,
                color=colors, edgecolor="white", height=0.4)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")

for bar, coef in zip(bars, coefs):
    offset = 0.05 if coef > 0 else -0.05
    ha     = "left" if coef > 0 else "right"
    plt.text(
        coef + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{coef:+.3f}",
        va="center", ha=ha, fontsize=10
    )

plt.xlabel("Coefficient (log-odds contribution)", fontsize=11)
plt.title("Feature Importance — Logistic Regression Weights", fontsize=12)
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*60)

---
## Step 16 — Predict for a New Weather Condition

We now use the trained model to predict whether golf will be played given a completely new weather observation.

**New weather condition:**

| Feature | Value | Encoded |
|---------|-------|--------|
| Temperature | Mild | Use encoder |
| Humidity | Normal | Use encoder |
| Windy | No | Use encoder |

**Pipeline for new input:**
1. Encode each text value using the same `LabelEncoder` fitted during Step 4
2. Reshape to `(1, 3)` — one row, three features
3. Scale using the same `scaler.transform()` — never re-fit
4. Get the hard prediction with `model.predict()`
5. Get the probability with `model.predict_proba()`
6. Decode the output label back to text

In [ ]:
# New weather condition prediction
new_temp     = "Mild"
new_humidity = "Normal"
new_windy    = "No"

# Encode using the same encoders from training
temp_encoded = encoders["Temperature"].transform([new_temp])[0]
hum_encoded  = encoders["Humidity"].transform([new_humidity])[0]
wind_encoded = encoders["Windy"].transform([new_windy])[0]

new_input        = np.array([[temp_encoded, hum_encoded, wind_encoded]])
new_input_scaled = scaler.transform(new_input)

# Predict
new_pred     = model.predict(new_input_scaled)
new_prob     = model.predict_proba(new_input_scaled)
pred_label   = encoders["PlayGolf"].inverse_transform(new_pred)[0]

print("New Weather Condition Prediction")
print("="*60)
print(f"Temperature    : {new_temp}")
print(f"Humidity       : {new_humidity}")
print(f"Windy          : {new_windy}")
print("="*60)
print(f"Encoded input  : {new_input[0]}")
print("="*60)
print(f"P(No)          : {new_prob[0][0]:.4f}")
print(f"P(Yes)         : {new_prob[0][1]:.4f}")
print("="*60)
print(f"Prediction     : PlayGolf = {pred_label}")
print("="*60)

---
## Step 16b — New Prediction Visualization

We visualize the predicted probability for the new weather condition as a horizontal probability bar. This makes the confidence of the prediction immediately readable — how far is the prediction from the 0.5 decision boundary?

In [ ]:
# Probability bar chart for new prediction
prob_no  = new_prob[0][0]
prob_yes = new_prob[0][1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Probability bar
bars = axes[0].barh(
    ["P(No)", "P(Yes)"],
    [prob_no, prob_yes],
    color=["tomato", "steelblue"],
    edgecolor="white", height=0.4
)
axes[0].axvline(0.5, color="black", linestyle="--",
                linewidth=1.5, label="Decision boundary (0.5)")
for bar, val in zip(bars, [prob_no, prob_yes]):
    axes[0].text(
        val + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", fontsize=11
    )
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Probability", fontsize=11)
axes[0].set_title(
    f"New Prediction: Temp={new_temp}, Humidity={new_humidity}, Windy={new_windy}",
    fontsize=11
)
axes[0].legend(fontsize=9)
axes[0].grid(True, axis="x", linestyle="--", alpha=0.5)

# Right: All test predictions probability comparison
x_pos     = range(len(y_prob_yes))
bar_colors = ["steelblue" if p >= 0.5 else "tomato" for p in y_prob_yes]
axes[1].bar(x_pos, y_prob_yes, color=bar_colors,
            edgecolor="white", width=0.5)
axes[1].axhline(0.5, color="black", linestyle="--",
                linewidth=1.5, label="Decision boundary (0.5)")
axes[1].set_xticks(list(x_pos))
axes[1].set_xticklabels(
    [f"S{i+1}\n({actual_text[i]})" for i in range(len(y_prob_yes))],
    fontsize=9
)
for i, (pos, prob) in enumerate(zip(x_pos, y_prob_yes)):
    axes[1].text(pos, prob + 0.02, f"{prob:.2f}",
                 ha="center", fontsize=9)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("P(PlayGolf = Yes)", fontsize=11)
axes[1].set_title("Test Set — Predicted Probabilities", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.suptitle(
    f"Final Prediction: PlayGolf = {pred_label}  |  P(Yes) = {prob_yes:.4f}",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()
print("="*60)

---
## Pipeline Summary

| Step | Action | Output |
|------|--------|--------|
| 1 | Import libraries | All tools including classification metrics |
| 2 | Create dataset | 10 weather observations, 4 columns |
| 3 | EDA — 4 bar charts | Class and feature distributions |
| 3b | Play rate per category | Which conditions lead to playing most |
| 4 | Label Encoding | All text columns converted to integers |
| 5 | Descriptive statistics and missing values | Encoded data quality confirmed |
| 6 | Feature and target selection | X shape (10,3), Y shape (10,) |
| 7 | Stratified train-test split 80/20 | 8 training, 2 test with class balance |
| 8 | StandardScaler | 3 features normalized |
| 9 | LogisticRegression trained | 3 weights + intercept + equation printed |
| 10 | Predictions + probabilities | Hard labels and P(Yes) per test sample |
| 11 | Accuracy, Precision, Recall, F1 | All 4 classification metrics |
| 12 | Confusion Matrix | TN, FP, FN, TP heatmap |
| 13 | Classification Report | Per-class and averaged metrics |
| 14 | ROC Curve + AUC Score | Classifier quality across all thresholds |
| 15 | Feature importance bar chart | Which weather factor matters most |
| 16 | New weather prediction | Mild, Normal, No Wind — prediction + probabilities |
| 16b | Probability visualization | Horizontal bar + test set probability comparison |

---

## Key Differences — Regression vs Classification

| Aspect | Linear Regression (Notebooks 01-02) | Logistic Regression (This Notebook) |
|--------|-------------------------------------|--------------------------------------|
| Output | Continuous number | Probability 0 to 1, then class label |
| Encoding needed | No (features were numeric) | Yes — all features were text |
| Loss function | MSE | Binary Cross-Entropy |
| Evaluation | MSE, RMSE, MAE, R2 | Accuracy, Precision, Recall, F1, AUC |
| Key extra plots | Regression line | ROC Curve, Confusion Matrix |
| Prediction method | `predict()` only | `predict()` + `predict_proba()` |

---

**Next notebook:** `multiclass_classification.ipynb` — Classifying more than 2 classes